# Previous Findings
* Removing all null values results in huge loss of information, about 55% of info loss, which means based on previous percentages of missing data found which totals 67% of null values, we can say that only 12% of null values exist in common rows, others are independent.
* Therefore this approach is not at all fruitfull and should be avoided.
* If dropping by null values, then considering the columns with least null values or solid logic in this domain will help to simplify the process a bit.

# Imports

In [1]:
import pandas as pd
import numpy as np

# Data Import

In [2]:
df = pd.read_csv("dirty_financial_transactions.csv")
df.head()

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,-5.0,$420.21,pay pal,NaN
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.34202525395585,creditcard,Pending
2,T0003,2025-02-30,C2919,Tablet,-4.0,810.9930123946459,credit card,completed
3,T0004,2020-08-17,C3009,Tab,-7.0,868.6083413217348,PayPal,Pending
4,T0005,2025-02-30,C3488,Coffee Machine,-10.0,-763.1224490039416,PayPal,completed


# First three column inspection and rapairing

In [3]:
df = df.dropna(subset=["Transaction_ID","Customer_ID"]) # Since missing Transaction_IDs and Customer_IDs can't be generated and the percentage of null 
                                                        #values in this column is very low, so dropping nulls based on it will not affect much.
df = df.sort_values(by=["Transaction_Date"], ascending=True) # Sorting by date for forward filling missing values.
df = df.reset_index(drop=True)
df.Transaction_Date = pd.to_datetime(df.Transaction_Date, yearfirst=True, errors="coerce") #Converting to date format and handling any errors.
df.Transaction_Date = df.Transaction_Date.ffill() #Forward filling dates
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 90340 entries, 0 to 90339
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      90340 non-null  str           
 1   Transaction_Date    90340 non-null  datetime64[us]
 2   Customer_ID         90340 non-null  str           
 3   Product_Name        90340 non-null  str           
 4   Quantity            85855 non-null  float64       
 5   Price               60108 non-null  str           
 6   Payment_Method      90340 non-null  str           
 7   Transaction_Status  75320 non-null  str           
dtypes: datetime64[us](1), float64(1), str(6)
memory usage: 29.8 MB


In [4]:
df.head(5)

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T7685,2020-01-01,C257,Coffee Machine,9.0,945.8729673990683,credit card,complete
1,T46858,2020-01-01,C3054,Coffee Machine,-2.0,952.3268604215747,PayPal,Pending
2,T59619,2020-01-01,C4785,Smartphone,5.0,NaN,Credit Card,NaN
3,T22312,2020-01-01,C687,Tablet,973.0,848.4864726981726,Cash,NaN
4,T39156,2020-01-01,C4666,Coffee Machine,1.0,800.9409011686039,credit card,Failed


First three columns are good to go, had to remove the rows with null Transaction_IDs and Customer_IDs as they can't be generated and if done then that might mislead.
The date column was in string format so converted it to datetime format and forward filled missing values.<br>
Trackbacing any Transaction_ID	or Customer_ID is now easy, with just the column wise sorting and the whole data of a customer's orders with data is available.<br>
These three columns were also checked for any wrong str value or such, and all clear, didn't gave the code to keep it clean.

# 4th, 5th and 6th column inspection and repairing

In [5]:
df.Product_Name.value_counts().head(15) #Kept it short for clear view, else there are too many values

Product_Name
Tablet            16644
Smartphone        16509
Laptop            16495
Headphones        16381
Coffee Machine    16377
Lapt                328
Table               318
Tabl                315
Ta                  314
La                  314
Lapto               309
Tab                 305
T                   305
Lap                 299
L                   285
Name: count, dtype: int64

Looks like the Product_Name column has incomplete values as it can be seen above, but to get the actual and legit product names that are in this list, we can see the majority value lies in the first 5 product names, so if we assume that there are only 5 types of products only in this list and other are just the incomplete namings of these 5 products only then i guess we can reverse engineer the names by first letter and retrive this data for now, else the nan ones can be figured out later if we get to find some pattern related to other columns

In [6]:
product_map = {
    "T":"Tablet",
    "S":"Smartphone",
    "L":"Laptop",
    "H":"Headphones",
    "C":"Coffee Machine"
}

In [7]:
df['Product_Name'] = df['Product_Name'].str[0].str.upper().map(product_map)
print(df['Product_Name'].value_counts())

Product_Name
Tablet            18201
Smartphone        18143
Laptop            18030
Headphones        18008
Coffee Machine    17958
Name: count, dtype: int64


Looks like we have successfully repaired the Product_Name column with no null values, next the Quantity column

In [8]:
df.Quantity.value_counts().head(10) # No tring value found yet in numbers, and we can see some negative values which souldn't be there

Quantity
-9.0     2927
-1.0     2899
 9.0     2896
-10.0    2889
 2.0     2874
-8.0     2872
 7.0     2869
-4.0     2867
-5.0     2860
-7.0     2854
Name: count, dtype: int64

In [9]:
df.Quantity = pd.to_numeric(df.Quantity) # Without the error param we can successfully convert the quantity to numeric, meaning no str is there confirmed
df.Quantity.isna().value_counts() #Even the value matches with null and non null found compared to df.info() above

Quantity
False    85855
True      4485
Name: count, dtype: int64

In [10]:
df.Quantity = df.Quantity.abs() # this is done to get all values in positive, as quantity cant be negative and previously we had found some negative values here.

In [11]:
df.Quantity.value_counts().head(5) #So negative values issue solved with numeric coversion, now its time to get the missing ones

Quantity
9.0    5823
7.0    5723
4.0    5706
1.0    5697
5.0    5691
Name: count, dtype: int64

For the missing values in Quantity column, my approach will be to get the mean value of the quantity for that the product ordered by that customer throughout the timeline, and if we see that it's the first order of that customer on that product,then we put the mean value of product for all order quantities on that perticular product. Lets find the all over mean values for each product quantity first.

In [12]:
mean_qty_dict = df.groupby('Product_Name')['Quantity'].mean().to_dict()
mean_qty_dict

{'Coffee Machine': 185.71669595782075,
 'Headphones': 190.54692348223173,
 'Laptop': 186.6930802919708,
 'Smartphone': 187.48254335260117,
 'Tablet': 187.70750622142486}

In [13]:
# Calculate mean per Customer/Product combination
customer_product_means = df.groupby(['Customer_ID', 'Product_Name'])['Quantity'].transform('mean')

# First pass: Fill NaNs using the specific Customer's history
df['Quantity'] = df['Quantity'].fillna(customer_product_means)

# Second pass: If still NaN (first-time order), use the product_map dictionary
# map() looks up the Product_Name in your dict and fills the hole
df['Quantity'] = df['Quantity'].fillna(df['Product_Name'].map(mean_qty_dict))
df["Quantity"] = df["Quantity"].astype("int16") #No need of stiring in float, that takes too much of space and quantity can't be in fraction,
                                                # and highest quantity is three digit, so int16 is safe side 

In [14]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 90340 entries, 0 to 90339
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      90340 non-null  str           
 1   Transaction_Date    90340 non-null  datetime64[us]
 2   Customer_ID         90340 non-null  str           
 3   Product_Name        90340 non-null  str           
 4   Quantity            90340 non-null  int16         
 5   Price               60108 non-null  str           
 6   Payment_Method      90340 non-null  str           
 7   Transaction_Status  75320 non-null  str           
dtypes: datetime64[us](1), int16(1), str(6)
memory usage: 29.4 MB


Therefore the `Quantity` column is all good, and even converted to int16, since the max number here doesn't go beyond 4 digit of numbers.<br>
Next we have the `Price` column, this one has str values like "$" sign and negative numbers to handle.

In [15]:
df['Price'] = df['Price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False)  #Removing the $ sign from the numbers
df['Price'] = pd.to_numeric(df['Price'], downcast='float')  #Converting to numeric dtype
df['Price'] = df['Price'].abs()  #Solves the negative number problem

* Time to fill the null values, since there are about 30k of missing data, simply mean imputation doesn't makes sence. <br>
* I almost picked a avg per product approach like before, but i carefully overved the data and found out something interesting, <br>
**Say, someone orders 2 Tablets, and it costs about 743`$` which is ok, now someone orders 299 Tablets and the cost is 363`$` , isn't this some error or pattern?<br>
Like there is some bulk discount system applied or these are server broken i.e. `someone tried to order in bulk by somewhat illegally breaking in the pricing system to scam or something like that`. Some of these transactions are either pending or cancelled, so we can't really trust them but some are completed too so it's a situation that cannot be ignored anymore. Leaving this to be handled in Phase 3.**<br>
* For now i wanna fill the null values by taking the quartile wise mean of the quantity's price i.e, we get the null in a row, check which product, and then for that product, get all the quantity values, sort and see the row's value falls in which quartile, the for that quartile we get all the price and mean it and then put it.

In [16]:
# Create 4 Quartiles based on Quantity for EACH Product
# Using duplicates='drop' in case many rows have the same low quantity (like 1 or 2)
df['Qty_Bin'] = df.groupby('Product_Name')['Quantity'].transform(
    lambda x: pd.qcut(x, q=4, labels=['Small', 'Medium', 'Large', 'Bulk'], duplicates='drop')
)#We will keep this column for easy investigation processes for cyber team


# Calculate the average price for each Product at that specific Quantity Bin
bin_price_map = df.groupby(['Product_Name', 'Qty_Bin'])['Price'].transform('mean')

# Fill the NaN prices using the bin average
df['Price'] = df['Price'].fillna(bin_price_map)

# Final Safety Net: If a specific bin has NO price data, use the general Product mean
df['Price'] = df['Price'].fillna(df.groupby('Product_Name')['Price'].transform('mean'))
df.sample(5)  #Next to Payment_Method column

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status,Qty_Bin
86346,T9763,2025-02-01,C4328,Coffee Machine,243,955.732483,PayPal,Failed,Large
71484,T9406,2025-02-01,C4029,Tablet,943,741.330688,credit card,Completed,Bulk
80758,T81725,2025-02-01,C186,Coffee Machine,9,641.996826,pay pal,Pending,Large
63857,T96543,2025-02-01,C714,Headphones,4,517.010010,credit card,Completed,Small
82745,T75263,2025-02-01,C4742,Tablet,404,576.935242,PayPal,complete,Bulk


In [17]:
df.Payment_Method.value_counts()

Payment_Method
PayPal         25606
Credit Card    13067
creditcard     12990
pay pal        12979
credit card    12860
Cash           12838
Name: count, dtype: int64

In [18]:
df["Payment_Method"] = df["Payment_Method"].str.lower().str.replace(" ", "")
df.Payment_Method.value_counts() #Multi value problem solved, this column is good to go now

Payment_Method
creditcard    38917
paypal        38585
cash          12838
Name: count, dtype: int64

In [19]:
df.Transaction_Status.value_counts()

Transaction_Status
complete     15150
Failed       15148
completed    15075
Pending      15067
Completed    14880
Name: count, dtype: int64

In [20]:
df["Transaction_Status"] = df["Transaction_Status"].str.lower().replace({'complete': 'completed'})
df.Transaction_Status.value_counts() #This fixes the multi-value problem in the Transaction_Status column

Transaction_Status
completed    45105
failed       15148
pending      15067
Name: count, dtype: int64

In [21]:
mode_status = df.groupby(['Product_Name', 'Qty_Bin'], observed=True)['Transaction_Status'].transform(
    lambda x: x.mode()[0] if not x.mode().empty else "completed"
)
# Taking product and order size wise mode of payment status, since we encountered a spam detection 
# situation, so its better to do it this way than taking all over mean
# 3. Fill the NaN values
df['Transaction_Status'] = df['Transaction_Status'].fillna(mode_status)

In [22]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 90340 entries, 0 to 90339
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      90340 non-null  str           
 1   Transaction_Date    90340 non-null  datetime64[us]
 2   Customer_ID         90340 non-null  str           
 3   Product_Name        90340 non-null  str           
 4   Quantity            90340 non-null  int16         
 5   Price               90340 non-null  float32       
 6   Payment_Method      90340 non-null  str           
 7   Transaction_Status  90340 non-null  str           
 8   Qty_Bin             90340 non-null  category      
dtypes: category(1), datetime64[us](1), float32(1), int16(1), str(5)
memory usage: 25.5 MB


# Exporting the cleaned data

In [23]:
df.to_csv("cleaned_financial_data.csv", index=False)

# Conclusion
Well this was the most complex and messy data we ever tackeled in this journey, lot of things were broken and missing, yet we managed to get most of it back.<br>
We even detected some suspicion of scam transactions by looking the quantity size vs the price and transaction states too, if needed this will be analyzed further on the upcoming phases.
<br>Thank you for reading through this notebook